# BeeSpace 04 - ERA5-Land e C3S

Simulação de variáveis climáticas para interpretar os sinais das colmeias. Em operação, esta camada deve usar dados C3S/ERA5-Land.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
datas = pd.date_range("2026-05-01", periods=30, freq="D")
serie = pd.DataFrame({
    "data": datas,
    "temp_media": np.random.normal(25, 4, len(datas)),
    "precipitacao_mm": np.random.gamma(shape=1.5, scale=5, size=len(datas)),
    "umidade_solo": np.clip(np.random.normal(0.35, 0.10, len(datas)), 0.05, 0.75),
    "radiacao": np.random.normal(18, 3, len(datas)),
    "vento": np.random.normal(2.5, 0.8, len(datas))
})

serie.loc[20:, "precipitacao_mm"] *= 0.15
serie.loc[20:, "umidade_solo"] *= 0.65
serie.loc[20:, "temp_media"] += 4

serie["precipitacao_7d"] = serie["precipitacao_mm"].rolling(7, min_periods=1).sum()
serie["risco_climatico"] = np.where(
    (serie["temp_media"] > 32) & (serie["precipitacao_7d"] < 15) & (serie["umidade_solo"] < 0.25),
    "alto",
    np.where(
        (serie["temp_media"] > 30) | (serie["precipitacao_7d"] < 20) | (serie["umidade_solo"] < 0.30),
        "medio",
        "baixo"
    )
)
serie.tail()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(serie["data"], serie["temp_media"], marker="o")
plt.title("Temperatura média simulada")
plt.xlabel("Data")
plt.ylabel("Temperatura")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "era5_temperatura_simulada.png", dpi=150)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(serie["data"], serie["precipitacao_7d"], marker="o")
plt.title("Precipitação acumulada em 7 dias")
plt.xlabel("Data")
plt.ylabel("mm")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "era5_precipitacao_7d_simulada.png", dpi=150)
plt.show()

serie.to_csv(OUTPUT_DIR / "serie_climatica_beespace.csv", index=False)